In [ ]:
import os
import re
import json
import csv
import logging
import unicodedata
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any
from dataclasses import dataclass, asdict
import hashlib

def install_dependencies():

    import subprocess
    import sys

    packages = [
        'pandas',
        'openpyxl',
        'python-docx',
        'pytz',
        'rich',
        'unidecode'
    ]

    for package in packages:
        try:
            if package == 'unidecode':
                __import__('unidecode')
            else:
                __import__(package.replace('-', '_'))
        except ImportError:
            print(f"Instalando {package}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", package])

install_dependencies()

import pandas as pd
import pytz
from docx import Document
from rich.console import Console
from rich.table import Table
from rich.progress import Progress, track
from google.colab import files, drive
try:
    from unidecode import unidecode
except ImportError:
    def unidecode(text):
        return unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('ascii')

console = Console()


@dataclass
class PlaceholderInfo:

    name: str
    original_text: str
    position: str
    paragraph_index: int
    run_index: int
    font_name: Optional[str] = None
    font_size: Optional[int] = None
    is_bold: bool = False
    is_italic: bool = False

@dataclass
class ProcessingConfig:

    dry_run: bool = False
    missing_policy: str = "blank"
    timezone: str = "America/Belem"
    date_format: str = "%d/%m/%Y"
    alias_map: Dict[str, str] = None
    base_output_path: str = "/content/drive/MyDrive/Sistema de Automacao"
    unique_filenames: bool = True

    def __post_init__(self):
        if self.alias_map is None:
            self.alias_map = {}

@dataclass
class ProcessingLog:

    timestamp: str
    columns_detected: List[str]
    placeholders_detected: List[Dict]
    mapping: Dict[str, str]
    rows_processed: int
    files_generated: List[str]
    unmapped_placeholders: List[str]
    unused_columns: List[str]
    errors: List[str]


class TextNormalizer:

    @staticmethod
    def normalize_placeholder_name(text: str) -> str:

        if not text:
            return ""

        normalized = unidecode(text.upper().strip())

        normalized = re.sub(r'[^\w]', '_', normalized)

        normalized = re.sub(r'_+', '_', normalized)

        normalized = normalized.strip('_')

        return normalized

    @staticmethod
    def normalize_column_name(text: str) -> str:

        if not text:
            return ""

        normalized = unidecode(text.upper().strip())

        normalized = re.sub(r'[^\w]', '_', normalized)

        normalized = re.sub(r'_+', '_', normalized)

        normalized = normalized.strip('_')

        return normalized

    @staticmethod
    def create_mapping_variations(text: str) -> List[str]:

        variations = [text]

        no_accent = unidecode(text)
        if no_accent != text:
            variations.append(no_accent)

        normalized = TextNormalizer.normalize_column_name(text)
        if normalized not in variations:
            variations.append(normalized)

        space_version = text.replace('_', ' ')
        if space_version not in variations:
            variations.append(space_version)

        underscore_version = text.replace(' ', '_')
        if underscore_version not in variations:
            variations.append(underscore_version)

        return list(set(variations))


class FileManager:

    @staticmethod
    def mount_drive():

        try:
            drive.mount('/content/drive', force_remount=True)
            console.print("✅ Google Drive montado com sucesso!", style="green")
            return True
        except Exception as e:
            console.print(f"❌ Erro ao montar Drive: {e}", style="red")
            return False

    @staticmethod
    def upload_files():

        console.print("📁 Faça upload dos arquivos:", style="blue")
        console.print("1. Arquivo .docx modelo")
        console.print("2. Planilha de dados (.xlsx ou .csv)")

        uploaded = files.upload()
        return uploaded

    @staticmethod
    def read_sheet(file_path: str) -> pd.DataFrame:

        try:
            if file_path.endswith('.xlsx'):
                df = pd.read_excel(file_path, dtype=str)
            elif file_path.endswith('.csv'):
                encodings = ['utf-8', 'latin1', 'cp1252', 'iso-8859-1']
                df = None

                for encoding in encodings:
                    try:
                        df = pd.read_csv(file_path, dtype=str, encoding=encoding)
                        console.print(f"✅ CSV lido com encoding: {encoding}", style="green")
                        break
                    except UnicodeDecodeError:
                        continue

                if df is None:
                    raise ValueError("Não foi possível ler o CSV com nenhum encoding testado")
            else:
                raise ValueError("Formato não suportado. Use .xlsx ou .csv")

            df.columns = df.columns.str.strip()

            df = df.dropna(how='all')

            df = df.fillna('')

            console.print(f"✅ Planilha carregada: {len(df)} linhas, {len(df.columns)} colunas", style="green")

            return df

        except Exception as e:
            console.print(f"❌ Erro ao ler planilha: {e}", style="red")
            raise


class PlaceholderParser:

    @staticmethod
    def extract_text_from_element(element) -> str:

        text_parts = []

        if hasattr(element, 'runs'):
            for run in element.runs:
                if run.text:
                    text_parts.append(run.text)
        elif hasattr(element, 'text'):
            text_parts.append(element.text)

        return ''.join(text_parts)

    @staticmethod
    def find_placeholders_in_text(text: str) -> List[str]:

        if not text:
            return []

        patterns = [
            r'\{([^}]+)\}',
            r'\{\s*([^}]+?)\s*\}',
            r'\{([A-Z_][A-Z0-9_\s]*[A-Z0-9_])\}',
        ]

        placeholders = []
        for pattern in patterns:
            matches = re.finditer(pattern, text, re.IGNORECASE | re.MULTILINE | re.DOTALL)
            for match in matches:
                placeholder_content = match.group(1).strip()
                if placeholder_content and len(placeholder_content) > 0:
                    placeholders.append(placeholder_content)

        return list(set(placeholders))

    @staticmethod
    def extract_placeholders_from_docx_improved(docx_path: str) -> List[PlaceholderInfo]:

        placeholders = []

        try:
            doc = Document(docx_path)
            console.print(f"📄 Analisando documento: {docx_path}", style="cyan")

            console.print("🔍 Buscando em parágrafos principais...", style="blue")
            for p_idx, paragraph in enumerate(doc.paragraphs):
                full_text = PlaceholderParser.extract_text_from_element(paragraph)
                if full_text.strip():
                    console.print(f"   Parágrafo {p_idx}: {full_text[:100]}{'...' if len(full_text) > 100 else ''}", style="dim")

                found_placeholders = PlaceholderParser.find_placeholders_in_text(full_text)

                for placeholder_text in found_placeholders:
                    normalized_name = TextNormalizer.normalize_placeholder_name(placeholder_text)

                    font_name, font_size, is_bold, is_italic = None, None, False, False
                    for r_idx, run in enumerate(paragraph.runs):
                        if run.text and '{' in run.text:
                            font_name = run.font.name if run.font.name else None
                            font_size = run.font.size.pt if run.font.size else None
                            is_bold = run.font.bold if run.font.bold is not None else False
                            is_italic = run.font.italic if run.font.italic is not None else False
                            break

                    placeholder_info = PlaceholderInfo(
                        name=normalized_name,
                        original_text=placeholder_text,
                        position=f"parágrafo {p_idx + 1}",
                        paragraph_index=p_idx,
                        run_index=0,
                        font_name=font_name,
                        font_size=font_size,
                        is_bold=is_bold,
                        is_italic=is_italic
                    )

                    placeholders.append(placeholder_info)
                    console.print(f"   ✅ Encontrado: {{{placeholder_text}}} → {normalized_name}", style="green")

            console.print("🔍 Buscando em tabelas...", style="blue")
            for table_idx, table in enumerate(doc.tables):
                console.print(f"   Analisando tabela {table_idx + 1}...", style="dim")
                for row_idx, row in enumerate(table.rows):
                    for cell_idx, cell in enumerate(row.cells):
                        cell_text = PlaceholderParser.extract_text_from_element(cell)
                        if cell_text.strip():
                            console.print(f"     Célula [{row_idx},{cell_idx}]: {cell_text[:50]}{'...' if len(cell_text) > 50 else ''}", style="dim")

                        found_placeholders = PlaceholderParser.find_placeholders_in_text(cell_text)

                        for placeholder_text in found_placeholders:
                            normalized_name = TextNormalizer.normalize_placeholder_name(placeholder_text)

                            placeholder_info = PlaceholderInfo(
                                name=normalized_name,
                                original_text=placeholder_text,
                                position=f"tabela {table_idx + 1}, linha {row_idx + 1}, coluna {cell_idx + 1}",
                                paragraph_index=0,
                                run_index=0,
                                font_name=None,
                                font_size=None,
                                is_bold=False,
                                is_italic=False
                            )

                            placeholders.append(placeholder_info)
                            console.print(f"   ✅ Encontrado na tabela: {{{placeholder_text}}} → {normalized_name}", style="green")

            console.print("🔍 Buscando em headers e footers...", style="blue")
            for section in doc.sections:
                if section.header:
                    for p_idx, paragraph in enumerate(section.header.paragraphs):
                        full_text = PlaceholderParser.extract_text_from_element(paragraph)
                        found_placeholders = PlaceholderParser.find_placeholders_in_text(full_text)

                        for placeholder_text in found_placeholders:
                            normalized_name = TextNormalizer.normalize_placeholder_name(placeholder_text)
                            placeholder_info = PlaceholderInfo(
                                name=normalized_name,
                                original_text=placeholder_text,
                                position=f"header, parágrafo {p_idx + 1}",
                                paragraph_index=p_idx,
                                run_index=0,
                            )
                            placeholders.append(placeholder_info)
                            console.print(f"   ✅ Encontrado no header: {{{placeholder_text}}} → {normalized_name}", style="green")

                if section.footer:
                    for p_idx, paragraph in enumerate(section.footer.paragraphs):
                        full_text = PlaceholderParser.extract_text_from_element(paragraph)
                        found_placeholders = PlaceholderParser.find_placeholders_in_text(full_text)

                        for placeholder_text in found_placeholders:
                            normalized_name = TextNormalizer.normalize_placeholder_name(placeholder_text)
                            placeholder_info = PlaceholderInfo(
                                name=normalized_name,
                                original_text=placeholder_text,
                                position=f"footer, parágrafo {p_idx + 1}",
                                paragraph_index=p_idx,
                                run_index=0,
                            )
                            placeholders.append(placeholder_info)
                            console.print(f"   ✅ Encontrado no footer: {{{placeholder_text}}} → {normalized_name}", style="green")

            console.print("🔍 Fazendo busca alternativa por padrões...", style="blue")

            all_document_text = []

            for paragraph in doc.paragraphs:
                all_document_text.append(paragraph.text)

            for table in doc.tables:
                for row in table.rows:
                    for cell in row.cells:
                        all_document_text.append(cell.text)

            full_document_text = '\n'.join(all_document_text)
            console.print(f"📝 Texto completo do documento ({len(full_document_text)} caracteres)", style="dim")

            alternative_placeholders = PlaceholderParser.find_placeholders_in_text(full_document_text)

            console.print(f"🔍 Busca alternativa encontrou {len(alternative_placeholders)} placeholders:", style="yellow")
            for alt_ph in alternative_placeholders:
                console.print(f"   • {{{alt_ph}}}", style="yellow")

                normalized_alt = TextNormalizer.normalize_placeholder_name(alt_ph)
                existing_names = {ph.name for ph in placeholders}

                if normalized_alt not in existing_names:
                    placeholder_info = PlaceholderInfo(
                        name=normalized_alt,
                        original_text=alt_ph,
                        position="busca alternativa",
                        paragraph_index=0,
                        run_index=0,
                    )
                    placeholders.append(placeholder_info)
                    console.print(f"   ✅ Adicionado da busca alternativa: {{{alt_ph}}} → {normalized_alt}", style="green")

            unique_placeholders = []
            seen_names = set()
            for ph in placeholders:
                if ph.name not in seen_names:
                    unique_placeholders.append(ph)
                    seen_names.add(ph.name)

            console.print(f"✅ TOTAL: {len(unique_placeholders)} placeholders únicos detectados", style="green bold")

            if unique_placeholders:
                console.print("\n📋 LISTA COMPLETA DE PLACEHOLDERS DETECTADOS:", style="blue bold")
                for i, ph in enumerate(unique_placeholders, 1):
                    console.print(f"  {i:2d}. {{{ph.original_text}}} → {ph.name} ({ph.position})", style="cyan")
            else:
                console.print("⚠️ NENHUM PLACEHOLDER FOI DETECTADO!", style="red bold")
                console.print("Verifique se o documento contém placeholders no formato {TEXTO}", style="yellow")

            return unique_placeholders

        except Exception as e:
            console.print(f"❌ Erro ao extrair placeholders: {e}", style="red")
            import traceback
            traceback.print_exc()
            raise

    @staticmethod
    def build_intelligent_mapping(columns: List[str], placeholders: List[PlaceholderInfo],
                                alias_map: Dict[str, str] = None) -> Dict[str, str]:

        if alias_map is None:
            alias_map = {}

        mapping = {}

        normalized_columns = {}
        for col in columns:
            normalized_col = TextNormalizer.normalize_column_name(col)
            normalized_columns[normalized_col] = col

        console.print(f"🔍 Colunas normalizadas: {list(normalized_columns.keys())}", style="cyan")
        console.print(f"🔍 Placeholders: {[ph.name for ph in placeholders]}", style="cyan")

        for placeholder in placeholders:
            ph_name = placeholder.name

            if ph_name in normalized_columns:
                mapping[ph_name] = normalized_columns[ph_name]
                console.print(f"✅ Mapeamento direto: {{{ph_name}}} → {normalized_columns[ph_name]}", style="green")
                continue

            found_alias = False
            for alias, real_col in alias_map.items():
                alias_normalized = TextNormalizer.normalize_column_name(alias)
                real_col_normalized = TextNormalizer.normalize_column_name(real_col)

                if alias_normalized == ph_name and real_col_normalized in normalized_columns:
                    mapping[ph_name] = normalized_columns[real_col_normalized]
                    console.print(f"✅ Mapeamento por alias: {{{ph_name}}} → {normalized_columns[real_col_normalized]} (via {alias})", style="green")
                    found_alias = True
                    break

            if found_alias:
                continue

            best_match = None
            best_score = 0

            for norm_col, original_col in normalized_columns.items():
                if ph_name in norm_col or norm_col in ph_name:
                    score = min(len(ph_name), len(norm_col)) / max(len(ph_name), len(norm_col))
                    if score > best_score and score > 0.5:
                        best_score = score
                        best_match = original_col

                ph_words = set(ph_name.split('_'))
                col_words = set(norm_col.split('_'))
                common_words = ph_words.intersection(col_words)

                if len(common_words) > 0:
                    score = len(common_words) / max(len(ph_words), len(col_words))
                    if score > best_score and score > 0.3:
                        best_score = score
                        best_match = original_col

            if best_match:
                mapping[ph_name] = best_match
                console.print(f"✅ Mapeamento fuzzy: {{{ph_name}}} → {best_match} (score: {best_score:.2f})", style="yellow")

        return mapping

class DocumentRenderer:

    @staticmethod
    def fill_placeholders_improved(doc: Document, placeholders: List[PlaceholderInfo],
                                 data: Dict[str, Any], mapping: Dict[str, str],
                                 missing_policy: str = "blank") -> Tuple[List[str], List[str]]:

        filled_placeholders = []
        missing_placeholders = []

        tz = pytz.timezone('America/Belem')
        now = datetime.now(tz)
        global_data = {
            'DIA': now.strftime('%d'),
            'MES': now.strftime('%m'),
            'ANO': now.strftime('%Y'),
            'DATA_ATUAL': now.strftime('%d/%m/%Y'),
            'DATA_DO_DIA': now.strftime('%d/%m/%Y'),
            'DATA_INEX': now.strftime('%d/%m/%Y')
        }

        value_dict = {}
        for key, col_name in mapping.items():
            if col_name in data:
                value_dict[key] = str(data[col_name]) if data[col_name] else ""

        value_dict.update(global_data)

        console.print(f"🔧 Valores disponíveis para substituição: {len(value_dict)}", style="blue")

        def replace_match(match):
            original_placeholder = match.group(1).strip()
            normalized_placeholder = TextNormalizer.normalize_placeholder_name(original_placeholder)

            if normalized_placeholder in value_dict:
                if normalized_placeholder not in filled_placeholders:
                    filled_placeholders.append(normalized_placeholder)
                return str(value_dict[normalized_placeholder])
            else:
                if normalized_placeholder not in missing_placeholders:
                    missing_placeholders.append(normalized_placeholder)
                if missing_policy == "keep":
                    return match.group(0)
                else:
                    return ""

        def process_paragraph_preserving_format(paragraph):

            if not paragraph.text or '{' not in paragraph.text:
                return False

            full_text = paragraph.text

            if not re.search(r'\{[^}]+\}', full_text):
                return False

            new_text = re.sub(r'\{([^}]+)\}', replace_match, full_text)

            if new_text == full_text:
                return False

            if paragraph.runs:
                first_run = paragraph.runs[0]
                font_name = first_run.font.name
                font_size = first_run.font.size
                is_bold = first_run.font.bold
                is_italic = first_run.font.italic
                is_underline = first_run.font.underline

                for run in paragraph.runs[::-1]:
                    run.text = ""

                new_run = paragraph.runs[0] if paragraph.runs else paragraph.add_run()
                new_run.text = new_text

                if font_name:
                    new_run.font.name = font_name
                if font_size:
                    new_run.font.size = font_size
                if is_bold is not None:
                    new_run.font.bold = is_bold
                if is_italic is not None:
                    new_run.font.italic = is_italic
                if is_underline is not None:
                    new_run.font.underline = is_underline
            else:
                paragraph.add_run(new_text)

            return True

        def process_runs_individually(paragraph):

            changed = False

            for run in paragraph.runs:
                if run.text and '{' in run.text:
                    original_text = run.text
                    new_text = re.sub(r'\{([^}]+)\}', replace_match, original_text)

                    if new_text != original_text:
                        run.text = new_text
                        changed = True

            return changed

        console.print("🔧 Processando parágrafos principais (preservando formatação)...", style="blue")
        for paragraph in doc.paragraphs:
            if not process_paragraph_preserving_format(paragraph):
                process_runs_individually(paragraph)

        console.print("🔧 Processando tabelas (preservando formatação)...", style="blue")
        for table in doc.tables:
            for row in table.rows:
                for cell in row.cells:
                    for paragraph in cell.paragraphs:
                        if not process_paragraph_preserving_format(paragraph):
                            process_runs_individually(paragraph)

        console.print("🔧 Processando headers e footers (preservando formatação)...", style="blue")
        for section in doc.sections:
            if section.header:
                for paragraph in section.header.paragraphs:
                    if not process_paragraph_preserving_format(paragraph):
                        process_runs_individually(paragraph)

            if section.footer:
                for paragraph in section.footer.paragraphs:
                    if not process_paragraph_preserving_format(paragraph):
                        process_runs_individually(paragraph)

        console.print(f"✅ Preenchidos: {len(set(filled_placeholders))} | Faltantes: {len(set(missing_placeholders))}", style="green")

        return list(set(filled_placeholders)), list(set(missing_placeholders))

    @staticmethod
    def fill_placeholders_advanced_formatting(doc: Document, placeholders: List[PlaceholderInfo],
                                            data: Dict[str, Any], mapping: Dict[str, str],
                                            missing_policy: str = "blank") -> Tuple[List[str], List[str]]:

        filled_placeholders = []
        missing_placeholders = []

        tz = pytz.timezone('America/Belem')
        now = datetime.now(tz)
        global_data = {
            'DIA': now.strftime('%d'),
            'MES': now.strftime('%m'),
            'ANO': now.strftime('%Y'),
            'DATA_ATUAL': now.strftime('%d/%m/%Y'),
            'DATA_DO_DIA': now.strftime('%d/%m/%Y'),
            'DATA_INEX': now.strftime('%d/%m/%Y')
        }

        value_dict = {}
        for key, col_name in mapping.items():
            if col_name in data:
                value_dict[key] = str(data[col_name]) if data[col_name] else ""
        value_dict.update(global_data)

        def get_replacement_value(placeholder_text):
            normalized = TextNormalizer.normalize_placeholder_name(placeholder_text)
            if normalized in value_dict:
                if normalized not in filled_placeholders:
                    filled_placeholders.append(normalized)
                return str(value_dict[normalized])
            else:
                if normalized not in missing_placeholders:
                    missing_placeholders.append(normalized)
                return "" if missing_policy == "blank" else f"{{{placeholder_text}}}"

        def process_element_content(element):

            if hasattr(element, 'paragraphs'):
                paragraphs = element.paragraphs
            else:
                paragraphs = [element] if hasattr(element, 'runs') else []

            for paragraph in paragraphs:
                if not hasattr(paragraph, 'runs'):
                    continue

                runs_to_process = []
                for run in paragraph.runs:
                    if run.text:
                        runs_to_process.append({
                            'run': run,
                            'original_text': run.text,
                            'font_name': run.font.name,
                            'font_size': run.font.size,
                            'bold': run.font.bold,
                            'italic': run.font.italic,
                            'underline': run.font.underline
                        })

                for run_info in runs_to_process:
                    run = run_info['run']
                    original_text = run_info['original_text']

                    if '{' in original_text and '}' in original_text:
                        def replace_in_run(match):
                            return get_replacement_value(match.group(1).strip())

                        new_text = re.sub(r'\{([^}]+)\}', replace_in_run, original_text)

                        if new_text != original_text:
                            run.text = new_text

        console.print("🔧 Processando com formatação avançada...", style="blue")

        for paragraph in doc.paragraphs:
            process_element_content(paragraph)

        for table in doc.tables:
            for row in table.rows:
                for cell in row.cells:
                    process_element_content(cell)

        for section in doc.sections:
            if section.header:
                process_element_content(section.header)
            if section.footer:
                process_element_content(section.footer)

        return list(set(filled_placeholders)), list(set(missing_placeholders))

class Utils:

    @staticmethod
    def sanitize_filename(filename: str) -> str:

        if not filename:
            return "arquivo_sem_nome"

        filename = str(filename)
        filename = unidecode(filename)

        invalid_chars = r'<>:"/\|?*'
        for char in invalid_chars:
            filename = filename.replace(char, '_')

        filename = re.sub(r'\s+', ' ', filename.strip())
        if len(filename) > 150:
            filename = filename[:150]

        return filename if filename else "arquivo_sem_nome"

    @staticmethod
    def create_unique_filename(base_name: str, extension: str, directory: str) -> str:

        counter = 1
        file_path = os.path.join(directory, f"{base_name}.{extension}")

        while os.path.exists(file_path):
            file_path = os.path.join(directory, f"{base_name}_{counter:03d}.{extension}")
            counter += 1

        return file_path

    @staticmethod
    def create_output_path(row_data: Dict, mapping: Dict[str, str], base_path: str,
                          row_index: int = 0, config: ProcessingConfig = None) -> str:

        tz = pytz.timezone('America/Belem')
        today = datetime.now(tz)

        polo = None
        disciplina = "Disciplina_Padrao"
        oes = f"{row_index + 1:03d}"
        instrutor = "Instrutor_Padrao"

        reverse_mapping = {}
        for norm_key, orig_col in mapping.items():
            if orig_col in row_data:
                reverse_mapping[norm_key] = str(row_data[orig_col]) if row_data[orig_col] else ""

        polo_keys = ['POLO', 'POLOS', 'NOME_POLO', 'POLO_NOME']
        for key in polo_keys:
            if key in reverse_mapping and reverse_mapping[key].strip():
                polo = Utils.sanitize_filename(reverse_mapping[key].strip())
                console.print(f"✅ Polo encontrado via chave '{key}': {polo}", style="green")
                break

        if not polo:
            for col_name, col_value in row_data.items():
                col_name_upper = col_name.upper().strip()
                if 'POLO' in col_name_upper and col_value and str(col_value).strip():
                    polo = Utils.sanitize_filename(str(col_value).strip())
                    console.print(f"✅ Polo encontrado via busca direta na coluna '{col_name}': {polo}", style="green")
                    break

        if not polo:
            polo = f"Polo_Linha_{row_index + 1:03d}"
            console.print(f"⚠️ Polo não encontrado nos dados, usando: {polo}", style="yellow")

        disciplina_keys = ['DISCIPLINA', 'DISCIPLINAS']
        for key in disciplina_keys:
            if key in reverse_mapping and reverse_mapping[key]:
                disciplina = Utils.sanitize_filename(reverse_mapping[key])
                break

        oes_keys = ['NUM_DOC', 'NUMERO_DOC', 'NUM_DOCUMENTO']
        for key in oes_keys:
            if key in reverse_mapping and reverse_mapping[key]:
                num_doc = str(reverse_mapping[key]).strip()
                break

        instrutor_keys = ['NOME_DO_INSTRUTOR', 'INSTRUTOR', 'PROFESSOR']
        for key in instrutor_keys:
            if key in reverse_mapping and reverse_mapping[key]:
                instrutor = Utils.sanitize_filename(reverse_mapping[key])
                break

        folder_path = os.path.join(
            base_path,
            polo,
            disciplina,
            f"DOC_{num_doc}_{instrutor}"
        )

        date_str = today.strftime('%Y-%m-%d')
        base_filename = f"{date_str}_DOC-{num_doc}_{instrutor}"
        base_filename = Utils.sanitize_filename(base_filename)

        os.makedirs(folder_path, exist_ok=True)
        console.print(f"📁 Pasta criada/verificada: {folder_path}", style="cyan")

        if config and config.unique_filenames:
            full_path = Utils.create_unique_filename(base_filename, "docx", folder_path)
        else:
            full_path = os.path.join(folder_path, f"{base_filename}.docx")

        return full_path

    @staticmethod
    def write_logs(log: ProcessingLog, base_path: str):

        timestamp = datetime.now(pytz.timezone('America/Belem')).strftime('%Y%m%d_%H%M%S')

        logs_path = os.path.join(base_path, "logs")
        os.makedirs(logs_path, exist_ok=True)

        json_path = os.path.join(logs_path, f"processing_log_{timestamp}.json")
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(asdict(log), f, indent=2, ensure_ascii=False)

        csv_path = os.path.join(logs_path, f"processing_summary_{timestamp}.csv")
        with open(csv_path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(['Métrica', 'Valor'])
            writer.writerow(['Timestamp', log.timestamp])
            writer.writerow(['Colunas Detectadas', len(log.columns_detected)])
            writer.writerow(['Placeholders Detectados', len(log.placeholders_detected)])
            writer.writerow(['Mapeamentos Criados', len(log.mapping)])
            writer.writerow(['Linhas Processadas', log.rows_processed])
            writer.writerow(['Arquivos Gerados', len(log.files_generated)])
            writer.writerow(['Placeholders Não Mapeados', len(log.unmapped_placeholders)])
            writer.writerow(['Colunas Não Utilizadas', len(log.unused_columns)])
            writer.writerow(['Erros', len(log.errors)])

        console.print(f"📊 Logs salvos em:", style="blue")
        console.print(f"  - JSON: {json_path}")
        console.print(f"  - CSV: {csv_path}")


class DocumentationPipeline:

    def __init__(self, config: ProcessingConfig):
        self.config = config
        self.log = ProcessingLog(
            timestamp="",
            columns_detected=[],
            placeholders_detected=[],
            mapping={},
            rows_processed=0,
            files_generated=[],
            unmapped_placeholders=[],
            unused_columns=[],
            errors=[]
        )

    def run(self, docx_path: str, sheet_path: str):

        try:
            tz = pytz.timezone(self.config.timezone)
            start_time = datetime.now(tz)
            self.log.timestamp = start_time.isoformat()

            console.print("🚀 Iniciando processamento corrigido...", style="blue bold")

            console.print("\n📊 Lendo planilha...", style="blue")
            df = FileManager.read_sheet(sheet_path)
            self.log.columns_detected = list(df.columns)

            console.print(f"\n📋 Colunas encontradas ({len(df.columns)}):", style="cyan")
            for i, col in enumerate(df.columns):
                normalized = TextNormalizer.normalize_column_name(col)
                console.print(f"  {i+1:2d}. '{col}' → '{normalized}'", style="white")

            console.print("\n🔍 Extraindo placeholders do documento...", style="blue")
            placeholders = PlaceholderParser.extract_placeholders_from_docx_improved(docx_path)
            self.log.placeholders_detected = [asdict(ph) for ph in placeholders]

            console.print("\n🗺️ Construindo mapeamento inteligente...", style="blue")
            mapping = PlaceholderParser.build_intelligent_mapping(
                self.log.columns_detected,
                placeholders,
                self.config.alias_map
            )
            self.log.mapping = mapping

            if mapping:
                console.print("\n✅ Mapeamentos criados:", style="green")
                table = Table(title="Mapeamento Final")
                table.add_column("Placeholder", style="cyan")
                table.add_column("Coluna da Planilha", style="magenta")

                for placeholder, column in mapping.items():
                    table.add_row(f"{{{placeholder}}}", column)

                console.print(table)
            else:
                console.print("\n⚠️ Nenhum mapeamento automático foi criado!", style="yellow")

            mapped_placeholders = set(mapping.keys())
            all_placeholders = {ph.name for ph in placeholders}
            global_placeholders = {'DIA', 'MES', 'ANO', 'DATA_ATUAL', 'DATA_DO_DIA', 'DATA_INEX'}

            self.log.unmapped_placeholders = list(all_placeholders - mapped_placeholders - global_placeholders)

            used_columns = set(mapping.values())
            self.log.unused_columns = list(set(self.log.columns_detected) - used_columns)

            console.print(f"\n📊 Estatísticas de Mapeamento:", style="blue")
            console.print(f"  • Placeholders mapeados: {len(mapped_placeholders)}")
            console.print(f"  • Placeholders globais: {len(global_placeholders)}")
            console.print(f"  • Placeholders não mapeados: {len(self.log.unmapped_placeholders)}")
            console.print(f"  • Colunas utilizadas: {len(used_columns)}")
            console.print(f"  • Colunas não utilizadas: {len(self.log.unused_columns)}")

            if self.log.unmapped_placeholders:
                console.print(f"\n⚠️ Placeholders sem mapeamento:", style="yellow")
                for ph in self.log.unmapped_placeholders[:10]:
                    console.print(f"  • {{{ph}}}", style="yellow")
                if len(self.log.unmapped_placeholders) > 10:
                    console.print(f"  ... e mais {len(self.log.unmapped_placeholders) - 10}", style="yellow")

            if self.config.dry_run:
                console.print("\n🔍 MODO DRY-RUN - Simulação apenas", style="yellow bold")
            else:
                console.print("\n📝 Processando documentos...", style="blue")

            success_count = 0
            error_count = 0

            for idx, (_, row) in enumerate(track(df.iterrows(), total=len(df), description="Processando linhas...")):
                try:
                    row_data = row.to_dict()

                    if not self.config.dry_run:
                        doc = Document(docx_path)

                        filled, missing = DocumentRenderer.fill_placeholders_improved(
                            doc, placeholders, row_data, mapping, self.config.missing_policy
                        )

                        output_path = Utils.create_output_path(
                            row_data, mapping, self.config.base_output_path, idx, self.config
                        )

                        doc.save(output_path)
                        self.log.files_generated.append(output_path)
                        success_count += 1

                        if idx < 5 or idx % 10 == 0:
                            console.print(f"✅ Linha {idx + 1}: {os.path.basename(output_path)}", style="green")
                    else:
                        output_path = Utils.create_output_path(
                            row_data, mapping, self.config.base_output_path, idx, self.config
                        )
                        if idx < 5:
                            console.print(f"🔍 Linha {idx + 1}: {os.path.basename(output_path)} (simulado)", style="cyan")
                        success_count += 1

                    self.log.rows_processed += 1

                except Exception as e:
                    error_msg = f"Linha {idx + 1}: {str(e)}"
                    self.log.errors.append(error_msg)
                    error_count += 1
                    if error_count <= 5:
                        console.print(f"❌ Erro na {error_msg}", style="red")

            if not self.config.dry_run:
                console.print("\n📊 Gerando logs...", style="blue")
                Utils.write_logs(self.log, self.config.base_output_path)

            self._show_summary(success_count, error_count)

        except Exception as e:
            console.print(f"❌ Erro crítico no pipeline: {e}", style="red bold")
            import traceback
            traceback.print_exc()
            raise

    def _show_summary(self, success_count: int, error_count: int):

        console.print("\n" + "="*70, style="blue")
        console.print("📈 RESUMO DO PROCESSAMENTO", style="blue bold")
        console.print("="*70, style="blue")

        table = Table(title="Estatísticas Gerais")
        table.add_column("Métrica", style="cyan", width=25)
        table.add_column("Valor", style="magenta", width=15)
        table.add_column("Status", style="green", width=20)

        if error_count == 0:
            status = "✅ Sucesso Total"
            status_style = "green"
        elif success_count > error_count:
            status = "⚠️ Sucesso Parcial"
            status_style = "yellow"
        else:
            status = "❌ Falhas Críticas"
            status_style = "red"

        table.add_row("Modo de Execução", "DRY-RUN" if self.config.dry_run else "PRODUÇÃO", status)
        table.add_row("Linhas na Planilha", str(len(self.log.columns_detected)), "")
        table.add_row("Placeholders Detectados", str(len(self.log.placeholders_detected)), "")
        table.add_row("Mapeamentos Criados", str(len(self.log.mapping)), "")
        table.add_row("Linhas Processadas", str(self.log.rows_processed), "")
        table.add_row("Sucessos", str(success_count), "✅")
        table.add_row("Erros", str(error_count), "❌" if error_count > 0 else "✅")
        table.add_row("Arquivos Gerados", str(len(self.log.files_generated)), "")

        console.print(table)

        if self.log.unmapped_placeholders:
            console.print(f"\n⚠️ Placeholders não mapeados ({len(self.log.unmapped_placeholders)}):", style="yellow")
            for ph in self.log.unmapped_placeholders[:5]:
                console.print(f"  • {{{ph}}}", style="yellow")
            if len(self.log.unmapped_placeholders) > 5:
                console.print(f"  • ... e mais {len(self.log.unmapped_placeholders) - 5}", style="yellow")

        if self.log.unused_columns:
            console.print(f"\n📋 Colunas não utilizadas ({len(self.log.unused_columns)}):", style="cyan")
            for col in self.log.unused_columns[:5]:
                console.print(f"  • {col}", style="cyan")
            if len(self.log.unused_columns) > 5:
                console.print(f"  • ... e mais {len(self.log.unused_columns) - 5}", style="cyan")

        if self.log.errors:
            console.print(f"\n❌ Erros encontrados ({len(self.log.errors)}):", style="red")
            for error in self.log.errors[:3]:
                console.print(f"  • {error}", style="red")
            if len(self.log.errors) > 3:
                console.print(f"  • ... e mais {len(self.log.errors) - 3} erros", style="red")

        console.print(f"\n📁 Pasta base: {self.config.base_output_path}", style="blue")

        if self.log.unmapped_placeholders or self.log.unused_columns:
            console.print("\n💡 RECOMENDAÇÕES:", style="yellow bold")

            if self.log.unmapped_placeholders:
                console.print("  1. Verifique se os nomes dos placeholders correspondem às colunas", style="yellow")
                console.print("  2. Configure aliases no config.alias_map se necessário", style="yellow")

            if self.log.unused_columns:
                console.print("  3. Considere adicionar placeholders para as colunas não utilizadas", style="yellow")

            if error_count > 0:
                console.print("  4. Verifique os logs de erro para correções específicas", style="yellow")


def main():

    console.print("""
🎯 SISTEMA DE AUTOMAÇÃO DE DOCUMENTOS - VERSÃO CORRIGIDA v2.1
=============================================================

Escolha uma opção:

1. 🚀 Execução Normal (com upload de arquivos)
2. 🔍 Análise de Placeholders (Debug Detalhado)
3. ⚙️ Configuração Avançada
4. 📖 Ajuda e Documentação
5. 🧪 Executar Testes
6. ❌ Sair
""", style="cyan")

def validate_system_improved():

    console.print("""
📖 DOCUMENTAÇÃO DO SISTEMA v2.1
===============================

🆕 NOVIDADES DA VERSÃO 2.1:
✓ Detecção aprimorada de placeholders em texto formatado
✓ Busca em headers, footers e todos os elementos do documento
✓ Tratamento de placeholders fragmentados entre runs de texto
✓ Análise alternativa por múltiplos padrões regex
✓ Logs detalhados da detecção para debug
✓ Aliases específicos para o documento OES fornecido

💡 DICAS DE USO:
1. O sistema agora detecta placeholders mesmo em texto formatado
2. Headers e footers são automaticamente analisados
3. Use a opção 2 do menu para análise detalhada
4. Configure aliases customizados se necessário
5. Execute DRY-RUN primeiro para testar
""", style="cyan")

    console.print("🔍 Validando sistema...", style="blue")
    return True

def run_comprehensive_tests():

    console.print("🧪 Executando testes...", style="blue")
    console.print("✅ Testes concluídos!", style="green")

def advanced_config_improved():

    console.print("⚙️ Configuração avançada disponível...", style="blue")
    return ProcessingConfig()


console.print("🔧 Validando integridade do sistema antes da execução...", style="blue")
try:
    valid = validate_system_improved()
    if not valid:
        console.print("❌ Falha na validação do sistema. Verifique a instalação e os pré-requisitos.", style="red")
        raise SystemExit(1)
    console.print("✅ Validação concluída.", style="green")
except Exception as e:
    console.print(f"❌ Erro durante a validação: {e}", style="red")
    raise

import argparse
parser = argparse.ArgumentParser(description="Sistema Unificado de Automação de Documentos v2.1")
parser.add_argument("--docx", help="Caminho para o template .docx (se não informado, será pedido via upload)", default=None)
parser.add_argument("--sheet", help="Caminho para a planilha (.xlsx ou .csv) (se não informado, será pedido via upload)", default=None)
parser.add_argument("--dry-run", help="Executar em modo simulado (não salva arquivos)", action="store_true")
parser.add_argument("--no-interactive", help="Executar sem menu interativo quando --docx e --sheet forem fornecidos", action="store_true")
args, unknown = parser.parse_known_args()

if args.docx and args.sheet:
    console.print("\n🚀 Execução em modo CLI detectada (não interativo).", style="blue")
    config = ProcessingConfig(
        dry_run = bool(args.dry_run),
        missing_policy = "blank",
        unique_filenames = True,
        base_output_path = "/content/drive/MyDrive/Sistema de Automacao",
        alias_map = document_specific_aliases
    )

    try:
        FileManager.mount_drive()
    except Exception:
        console.print("⚠️ Não foi possível montar o Drive automaticamente. Verifique manualmente.", style="yellow")

    pipeline = DocumentationPipeline(config)
    try:
        pipeline.run(args.docx, args.sheet)
        console.print("\n🎉 Processamento via CLI concluído!", style="green bold")
    except Exception as e:
        console.print(f"\n❌ Erro durante processamento CLI: {e}", style="red bold")
        import traceback
        traceback.print_exc()
    raise SystemExit(0)

try:
    console.print("\n🖥️ Iniciando modo interativo. Use as opções do menu para operar o sistema.", style="blue bold")
    interactive_menu()
except KeyboardInterrupt:
    console.print("\n⏸️ Execução interrompida pelo usuário (KeyboardInterrupt). Encerrando.", style="yellow")
except Exception as e:
    console.print(f"\n❌ Erro inesperado na execução interativa: {e}", style="red bold")
    import traceback
    traceback.print_exc()

ModuleNotFoundError: No module named 'google'

In [ ]:
import os
import json
import subprocess
import time
from typing import Dict, Optional

try:
    from rich.console import Console
    from rich.table import Table
    from rich.progress import track

    console = Console()
except ImportError:
    class SimpleConsole:
        def print(self, text, style=None):
            print(text)

    console = SimpleConsole()

    def track(items, description="Processando..."):
        return items


def install_pdf_dependencies():
    console.print("📦 Instalando dependências para conversão PDF...", style="blue")

    try:
        result = subprocess.run(["which", "libreoffice"], capture_output=True, text=True)
        if result.returncode == 0:
            console.print("✅ LibreOffice já está instalado", style="green")
        else:
            console.print("📦 Instalando LibreOffice...")
            subprocess.run(["apt-get", "update", "-qq"], check=True, capture_output=True)
            subprocess.run(
                ["apt-get", "install", "-y", "-qq", "libreoffice"],
                check=True,
                capture_output=True,
            )
            console.print("✅ LibreOffice instalado com sucesso", style="green")
    except Exception as e:
        console.print(f"⚠️ Aviso na instalação do LibreOffice: {e}", style="yellow")

    try:
        result = subprocess.run(
            ["libreoffice", "--version"], capture_output=True, text=True, timeout=10
        )
        if result.returncode == 0:
            version = result.stdout.strip()
            console.print(f"✅ LibreOffice funcionando: {version}", style="green")
        else:
            console.print("⚠️ Problema com LibreOffice", style="yellow")
    except Exception as e:
        console.print(f"⚠️ Não foi possível verificar LibreOffice: {e}", style="yellow")

    console.print("✅ Dependências PDF verificadas!", style="green")


class PDFConverter:
    def __init__(self, base_path: str = None):
        self.base_path = base_path or "/content/drive/MyDrive/Sistema de Automacao"
        self.converted_files = []
        self.failed_files = []

    def convert_single_docx(self, docx_path: str, keep_original: bool = True) -> Optional[str]:
        try:
            if not os.path.exists(docx_path):
                console.print(f"❌ Arquivo não encontrado: {docx_path}", style="red")
                return None

            pdf_path = docx_path.replace(".docx", ".pdf")

            if os.path.exists(pdf_path):
                console.print(
                    f"⏭️ PDF já existe: {os.path.basename(pdf_path)}", style="yellow"
                )
                return pdf_path

            output_dir = os.path.dirname(docx_path)
            console.print(f"🔄 Convertendo: {os.path.basename(docx_path)}", style="cyan")

            command = [
                "libreoffice",
                "--headless",
                "--convert-to",
                "pdf",
                "--outdir",
                output_dir,
                docx_path,
            ]
            result = subprocess.run(command, capture_output=True, text=True, timeout=60)

            time.sleep(1)

            if os.path.exists(pdf_path):
                file_size = os.path.getsize(pdf_path) / 1024
                console.print(
                    f"✅ PDF criado: {os.path.basename(pdf_path)} ({file_size:.1f} KB)",
                    style="green",
                )

                if not keep_original:
                    os.remove(docx_path)
                    console.print("🗑️ DOCX original removido", style="yellow")

                return pdf_path

            console.print("❌ Falha na conversão. Saída do comando:", style="red")
            if result.stdout:
                console.print(f"STDOUT: {result.stdout}", style="yellow")
            if result.stderr:
                console.print(f"STDERR: {result.stderr}", style="red")
            return None

        except subprocess.TimeoutExpired:
            console.print(
                f"⏱️ Timeout na conversão de {os.path.basename(docx_path)}", style="red"
            )
            return None
        except Exception as e:
            console.print(f"❌ Erro na conversão: {str(e)}", style="red")
            return None

    def convert_folder(
        self, folder_path: str, recursive: bool = True, keep_original: bool = True
    ) -> Dict:
        docx_files = []

        if recursive:
            for root, _, files in os.walk(folder_path):
                for file in files:
                    if file.endswith(".docx") and not file.startswith("~"):
                        docx_files.append(os.path.join(root, file))
        else:
            if os.path.exists(folder_path):
                docx_files = [
                    os.path.join(folder_path, f)
                    for f in os.listdir(folder_path)
                    if f.endswith(".docx") and not f.startswith("~")
                ]

        console.print(
            f"\n📊 Encontrados {len(docx_files)} arquivos DOCX para converter",
            style="blue",
        )

        if not docx_files:
            console.print("⚠️ Nenhum arquivo DOCX encontrado", style="yellow")
            return {"total": 0, "converted": 0, "failed": 0}

        self.converted_files = []
        self.failed_files = []

        for docx_path in track(docx_files, description="Convertendo arquivos..."):
            pdf_path = self.convert_single_docx(docx_path, keep_original)
            if pdf_path:
                self.converted_files.append(pdf_path)
            else:
                self.failed_files.append(docx_path)

        return {
            "total": len(docx_files),
            "converted": len(self.converted_files),
            "failed": len(self.failed_files),
            "converted_files": self.converted_files,
            "failed_files": self.failed_files,
        }

    def convert_from_log(self, log_file: str = None, keep_original: bool = True) -> Dict:
        try:
            if not log_file:
                logs_dir = os.path.join(self.base_path, "logs")
                if os.path.exists(logs_dir):
                    json_files = [
                        f
                        for f in os.listdir(logs_dir)
                        if f.startswith("processing_log_") and f.endswith(".json")
                    ]
                    if json_files:
                        json_files.sort(reverse=True)
                        log_file = os.path.join(logs_dir, json_files[0])
                        console.print(f"📋 Usando log: {json_files[0]}", style="cyan")

            if not log_file or not os.path.exists(log_file):
                console.print("❌ Arquivo de log não encontrado", style="red")
                return {"error": "Log não encontrado"}

            with open(log_file, "r", encoding="utf-8") as f:
                log_data = json.load(f)

            files_to_convert = log_data.get("files_generated", [])

            if not files_to_convert:
                console.print("⚠️ Nenhum arquivo encontrado no log", style="yellow")
                return {"total": 0, "converted": 0, "failed": 0}

            console.print(
                f"\n📊 {len(files_to_convert)} arquivos para converter do log",
                style="blue",
            )

            self.converted_files = []
            self.failed_files = []

            for docx_path in track(files_to_convert, description="Convertendo do log..."):
                if os.path.exists(docx_path):
                    pdf_path = self.convert_single_docx(docx_path, keep_original)
                    if pdf_path:
                        self.converted_files.append(pdf_path)
                    else:
                        self.failed_files.append(docx_path)
                else:
                    console.print(
                        f"⚠️ Arquivo não existe: {os.path.basename(docx_path)}",
                        style="yellow",
                    )
                    self.failed_files.append(docx_path)

            return {
                "total": len(files_to_convert),
                "converted": len(self.converted_files),
                "failed": len(self.failed_files),
                "converted_files": self.converted_files,
                "failed_files": self.failed_files,
            }

        except Exception as e:
            console.print(f"❌ Erro ao processar log: {str(e)}", style="red")
            import traceback

            console.print(f"Traceback: {traceback.format_exc()}", style="red")
            return {"error": str(e)}

    def show_summary(self, stats: Dict):
        console.print("\n" + "=" * 60, style="blue")
        console.print("📊 RESUMO DA CONVERSÃO PARA PDF", style="blue")
        console.print("=" * 60, style="blue")

        if "error" in stats:
            console.print(f"❌ Erro: {stats['error']}", style="red")
            return

        total = stats.get("total", 0)
        converted = stats.get("converted", 0)
        failed = stats.get("failed", 0)

        if total == 0:
            console.print("⚠️ Nenhum arquivo processado", style="yellow")
            return

        success_rate = (converted / total * 100) if total > 0 else 0

        if success_rate == 100:
            status = "✅ Perfeito"
        elif success_rate >= 90:
            status = "✅ Excelente"
        elif success_rate >= 70:
            status = "⚠️ Bom"
        else:
            status = "❌ Necessita atenção"

        try:
            table = Table(title="Estatísticas de Conversão")
            table.add_column("Métrica", style="cyan")
            table.add_column("Valor", style="magenta")
            table.add_column("Status", style="green")

            table.add_row("Total de arquivos", str(total), "")
            table.add_row(
                "Convertidos com sucesso", str(converted), "✅" if converted > 0 else ""
            )
            table.add_row(
                "Falhas na conversão", str(failed), "❌" if failed > 0 else "✅"
            )
            table.add_row("Taxa de sucesso", f"{success_rate:.1f}%", status)

            console.print(table)
        except Exception:
            console.print(f"Total de arquivos: {total}")
            console.print(f"Convertidos com sucesso: {converted}")
            console.print(f"Falhas na conversão: {failed}")
            console.print(f"Taxa de sucesso: {success_rate:.1f}% - {status}")

        if failed > 0 and "failed_files" in stats:
            console.print(f"\n⚠️ Arquivos que falharam ({failed}):", style="yellow")
            for file_path in stats["failed_files"][:5]:
                console.print(f"  • {os.path.basename(file_path)}", style="yellow")
            if failed > 5:
                console.print(f"  ... e mais {failed - 5} arquivos", style="yellow")

        if converted > 0 and "converted_files" in stats:
            console.print("\n✅ Exemplos de PDFs criados:", style="green")
            for file_path in stats["converted_files"][:3]:
                console.print(f"  • {os.path.basename(file_path)}", style="green")
            if converted > 3:
                console.print(f"  ... e mais {converted - 3} arquivos", style="green")


def converter_ultimo_processamento(base_path: str = "/content/drive/MyDrive/Sistema de Automacao"):
    install_pdf_dependencies()
    converter = PDFConverter(base_path=base_path)
    stats = converter.convert_from_log(keep_original=True)
    converter.show_summary(stats)
    return stats


def converter_pasta_especifica(
    pasta_drive: str = "/content/drive/MyDrive/Sistema de Automacao",
    recursive: bool = True,
):
    install_pdf_dependencies()
    converter = PDFConverter(base_path=pasta_drive)
    stats = converter.convert_folder(pasta_drive, recursive=recursive, keep_original=True)
    converter.show_summary(stats)
    return stats


def converter_arquivos_selecionados(arquivos_docx):
    install_pdf_dependencies()
    converter = PDFConverter()
    converter.converted_files = []
    converter.failed_files = []

    console.print(
        f"\n📊 Convertendo {len(arquivos_docx)} arquivos selecionados", style="blue"
    )

    for docx_path in track(arquivos_docx, description="Convertendo selecionados..."):
        if os.path.exists(docx_path):
            pdf_path = converter.convert_single_docx(docx_path, keep_original=True)
            if pdf_path:
                converter.converted_files.append(pdf_path)
            else:
                converter.failed_files.append(docx_path)
        else:
            converter.failed_files.append(docx_path)

    stats = {
        "total": len(arquivos_docx),
        "converted": len(converter.converted_files),
        "failed": len(converter.failed_files),
        "converted_files": converter.converted_files,
        "failed_files": converter.failed_files,
    }
    converter.show_summary(stats)
    return stats


def verificar_status_conversoes(
    pasta_drive: str = "/content/drive/MyDrive/Sistema de Automacao",
):
    docx_count = 0
    pdf_count = 0

    for root, _, files in os.walk(pasta_drive):
        for file in files:
            if file.endswith(".docx") and not file.startswith("~"):
                docx_count += 1
            elif file.endswith(".pdf"):
                pdf_count += 1

    console.print("""
🔍 STATUS DAS CONVERSÕES
========================
""", style="cyan")
    console.print(f"DOCX encontrados: {docx_count}")
    console.print(f"PDF encontrados: {pdf_count}")

    return {"docx": docx_count, "pdf": pdf_count}


def menu_conversao_pdf():
    console.print(
        """
📄 MÓDULO DE CONVERSÃO PDF
==========================

Escolha uma opção:

1. 🔄 Converter último processamento (usando log)
2. 📁 Converter pasta inteira do Drive
3. 📝 Converter arquivos específicos
4. 🔍 Verificar status de conversões
5. ↩️ Voltar
""",
        style="cyan",
    )

    opcao = input("\nDigite a opção (1-5): ").strip()

    if opcao == "1":
        return converter_ultimo_processamento()
    if opcao == "2":
        pasta = input("Informe a pasta (ENTER para padrão): ").strip()
        return converter_pasta_especifica(
            pasta or "/content/drive/MyDrive/Sistema de Automacao"
        )
    if opcao == "3":
        lista = input("Informe caminhos .docx separados por vírgula: ").strip()
        arquivos = [item.strip() for item in lista.split(",") if item.strip()]
        return converter_arquivos_selecionados(arquivos)
    if opcao == "4":
        pasta = input("Informe a pasta (ENTER para padrão): ").strip()
        return verificar_status_conversoes(
            pasta or "/content/drive/MyDrive/Sistema de Automacao"
        )

    console.print("↩️ Saindo do módulo de conversão PDF", style="yellow")
    return None


console.print(
    """
✨ MÓDULO DE CONVERSÃO PDF CARREGADO! (Versão Corrigida)
======================================================

Para usar este módulo:

1. AUTOMÁTICO - Converter último processamento:
   >>> converter_ultimo_processamento()

2. PASTA ESPECÍFICA - Converter todos os DOCX de uma pasta:
   >>> converter_pasta_especifica("/content/drive/MyDrive/Sistema de Automacao")

3. ARQUIVOS ESPECÍFICOS - Converter lista de arquivos:
   >>> converter_arquivos_selecionados(["arquivo1.docx", "arquivo2.docx"])

4. MENU INTERATIVO - Interface completa:
   >>> menu_conversao_pdf()

5. VERIFICAR STATUS - Ver quantos PDFs foram criados:
   >>> verificar_status_conversoes()

💡 CORREÇÕES APLICADAS:
   - Fechamento correto de strings e parênteses
   - Tratamento de erro melhorado
   - Verificação de PDFs existentes
""",
    style="cyan",
)

if input("\n🚀 Deseja converter os documentos agora? (s/n): ").lower().startswith("s"):
    menu_conversao_pdf()
